# 06 - Comparison

This notebook compares all regularization experiments under the same controlled setup.

## Goal
The goal is to summarize how each method affects:
- training loss
- validation loss
- test loss
- training accuracy
- validation accuracy
- test accuracy
- generalization gap

## Compared Experiments
- Baseline
- Dropout
- Early Stopping
- BatchNorm
- Dropout + BatchNorm
- L2 (MAP Gaussian Prior) + Dropout

In [ ]:
# Import shared data loading, training, and summary utilities
from src.data import get_dataloaders
from src.train_eval import run_experiment
from src.plots import make_summary_table, save_summary_table, plot_all_experiments_vs_baseline
import os
import json

In [ ]:
# Load the fixed train / validation / test split
train_loader, val_loader, test_loader = get_dataloaders()

In [ ]:
def load_saved_results(results_dir="results/histories"):
    results = {}

    for filename in os.listdir(results_dir):
        if filename.endswith(".json"):
            key = filename.replace(".json", "")
            with open(os.path.join(results_dir, filename), "r", encoding="utf-8") as f:
                results[key] = json.load(f)

    return results

In [ ]:
# Store all experiment outputs in a single dictionary
results = load_saved_results()

In [ ]:
# Run the baseline experiment
results["baseline"] = run_experiment(
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    experiment_name="Baseline",
    hidden_sizes=[1024, 512, 256],
    use_dropout=False,
    use_batchnorm=False,
    use_early_stopping=False,
    learning_rate=0.01,
    weight_decay=0.0,
    use_explicit_l2=False,
    max_epochs=50,
    seed=42
)

In [ ]:
# Run the dropout experiment
results["dropout"] = run_experiment(
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    experiment_name="Dropout",
    hidden_sizes=[1024, 512, 256],
    use_dropout=True,
    dropout_p=0.5,
    use_batchnorm=False,
    use_early_stopping=False,
    learning_rate=0.01,
    weight_decay=0.0,
    use_explicit_l2=False,
    max_epochs=50,
    seed=42
)

In [ ]:
# Run the early stopping experiment
results["early_stopping"] = run_experiment(
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    experiment_name="EarlyStopping",
    hidden_sizes=[1024, 512, 256],
    use_dropout=False,
    use_batchnorm=False,
    use_early_stopping=True,
    patience=5,
    min_delta=1e-4,
    learning_rate=0.01,
    weight_decay=0.0,
    use_explicit_l2=False,
    max_epochs=50,
    seed=42
)

In [ ]:
# Run the batch normalization experiment
results["batchnorm"] = run_experiment(
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    experiment_name="BatchNorm",
    hidden_sizes=[1024, 512, 256],
    use_dropout=False,
    use_batchnorm=True,
    use_early_stopping=False,
    learning_rate=0.01,
    weight_decay=0.0,
    use_explicit_l2=False,
    max_epochs=50,
    seed=42
)

In [ ]:
# Run the combined dropout + batch normalization experiment
results["dropout_batchnorm"] = run_experiment(
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    experiment_name="Dropout + BatchNorm",
    hidden_sizes=[1024, 512, 256],
    use_dropout=True,
    dropout_p=0.5,
    use_batchnorm=True,
    use_early_stopping=False,
    learning_rate=0.01,
    weight_decay=0.0,
    use_explicit_l2=False,
    max_epochs=50,
    seed=42
)

In [ ]:
# Run the dropout + explicit L2 experiment with the MAP Gaussian prior interpretation
results["l2_dropout"] = run_experiment(
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    experiment_name="L2 (MAP Gaussian Prior) + Dropout",
    hidden_sizes=[1024, 512, 256],
    use_dropout=True,
    dropout_p=0.5,
    use_batchnorm=False,
    use_early_stopping=False,
    learning_rate=0.01,
    weight_decay=1e-4,
    use_explicit_l2=True,
    max_epochs=50,
    seed=42
)

In [ ]:
# Build and display the final summary table
summary_df = make_summary_table(results)
print(summary_df)

In [ ]:
# Save the summary table as a CSV file
save_summary_table(summary_df, "results/tables/experiment_summary.csv")

In [ ]:
# Display the final summary DataFrame
summary_df

In [ ]:
import os

# Create figure output directory
os.makedirs("results/figures", exist_ok=True)

# Baseline vs all experiments - validation loss
plot_all_experiments_vs_baseline(
    results=results,
    metric_key="val_loss",
    ylabel="Validation Loss",
    title="All Experiments vs Baseline - Validation Loss",
    save_path="results/figures/all_vs_baseline_val_loss.png"
)

# Baseline vs all experiments - test loss
plot_all_experiments_vs_baseline(
    results=results,
    metric_key="test_loss",
    ylabel="Test Loss",
    title="All Experiments vs Baseline - Test Loss",
    save_path="results/figures/all_vs_baseline_test_loss.png"
)

# Baseline vs all experiments - validation accuracy
plot_all_experiments_vs_baseline(
    results=results,
    metric_key="val_acc",
    ylabel="Validation Accuracy (%)",
    title="All Experiments vs Baseline - Validation Accuracy",
    save_path="results/figures/all_vs_baseline_val_acc.png"
)

# Baseline vs all experiments - test accuracy
plot_all_experiments_vs_baseline(
    results=results,
    metric_key="test_acc",
    ylabel="Test Accuracy (%)",
    title="All Experiments vs Baseline - Test Accuracy",
    save_path="results/figures/all_vs_baseline_test_acc.png"
)

# Baseline vs all experiments - generalization gap
plot_all_experiments_vs_baseline(
    results=results,
    metric_key=None,
    ylabel="Train - Validation Gap (%)",
    title="All Experiments vs Baseline - Generalization Gap",
    save_path="results/figures/all_vs_baseline_gap.png",
    gap_mode=True
)